In [0]:
from pyspark.sql.functions import to_date, dayofweek, month, sum as spark_sum, count

fact = spark.table("retail_project.silver.order_items_enriched") \
    .withColumn("order_date", to_date("order_purchase_timestamp"))

daily_category_sales = (
    fact.groupBy("order_date", "product_category_name_english")
    .agg(
        spark_sum("price").alias("total_sales"),
        count("order_id").alias("order_count")
    )
    .withColumn("day_of_week", dayofweek("order_date"))
    .withColumn("month", month("order_date"))
)

daily_category_sales.write.format("delta").mode("overwrite") \
    .saveAsTable("retail_project.gold.daily_category_sales")

print(f"Daily category sales rows: {daily_category_sales.count()}")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, avg

w = Window.partitionBy("product_category_name_english").orderBy("order_date")

features = (
    daily_category_sales
    .withColumn("lag_1_sales", lag("total_sales", 1).over(w))
    .withColumn("lag_7_sales", lag("total_sales", 7).over(w))
    .withColumn("rolling_avg_7", avg("total_sales").over(w.rowsBetween(-7, -1)))
    .na.drop(subset=["lag_1_sales", "lag_7_sales", "rolling_avg_7"])
)

features.write.format("delta").mode("overwrite") \
    .saveAsTable("retail_project.gold.demand_forecast_features")

print(f"Feature table rows: {features.count()}")